# 02 — Critères, indice et étiquettes radar

Ce notebook explique **comment on passe des couches brutes à l'indice**, et comment sont fabriquées les étiquettes d'eau Sentinel-1.
Il lit les résultats des scripts (`src/features/susceptibility.py`, `src/features/flood_labels.py`) et réutilise leurs fonctions : le code n'est pas dupliqué.

**Rappel** : l'indice est **relatif** (un rang entre 0 et 1), pas une prédiction d'inondation.

In [9]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

# Racine du projet : le notebook peut être lancé depuis notebooks/ ou depuis la racine.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
PROC = ROOT / "data" / "processed"
OUT = ROOT / "outputs"
plt.rcParams["figure.dpi"] = 110


def load(name, folder=PROC):
    """Lit un raster (bande 1) en float, les valeurs « sans donnée » deviennent NaN."""
    with rasterio.open(folder / name) as src:
        arr = src.read(1).astype("float64")
        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan
    return arr


def show(arr, title, cmap="viridis", vmin=None, vmax=None, ax=None):
    ax = ax or plt.gca()
    im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=10)
    ax.set_axis_off()
    plt.colorbar(im, ax=ax, shrink=0.75)

from src.features import susceptibility as susc

## 1. Pourquoi normaliser par le rang et pas par min-max

Test : on construit l'indice avec les deux méthodes et on regarde la distribution.

In [10]:
raw = {name: load(path.name) for name, path in susc.PATHS.items()}
dem = load("dem_bamako_utm.tif")
valid = ~np.isnan(dem)
for k in raw:
    valid &= ~np.isnan(raw[k])


def minmax_low_is_risky(a):
    v = a[valid]
    return 1 - (a - v.min()) / (v.max() - v.min())


def minmax(name, a):
    return 1 - minmax_low_is_risky(a) if name in susc.HIGH_IS_RISKY else minmax_low_is_risky(a)


score_minmax = np.mean([minmax(k, raw[k]) for k in susc.CRITERIA], axis=0)
layers = {k: susc.normalize(k, raw[k]) for k in susc.CRITERIA}
score_rank = susc.weighted_score(layers, susc.SCENARIOS[susc.MAIN_SCENARIO])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, (name, s) in zip(axes, {"min-max (abandonné)": score_minmax, "rang (retenu)": score_rank}.items()):
    ax.hist(s[valid], bins=60, color="#c0504d" if "min-max" in name else "#4a7ebb")
    ax.axvline(0.7, color="k", ls="--", lw=0.8)
    ax.set_title(f"{name} : {100 * np.mean(s[valid] > 0.7):.0f} % des pixels > 0,7")
plt.tight_layout()

**Lecture.** Avec min-max, la plupart des pixels se retrouvent « très exposés » : l'échelle sature et ne dit plus rien. Avec le rang, 0,5 est la médiane et 0,7 signifie « parmi les ~30 % les plus exposés ». Contrepartie : on ne garde que l'ordre, pas l'écart réel (10 m ou 50 m au-dessus du fleuve comptent par leur rang).

## 2. Les quatre critères normalisés

In [11]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
titles = {"height": "Hauteur au-dessus du Niger", "distance": "Distance au cours d'eau",
          "slope": "Pente", "twi": "TWI"}
for ax, k in zip(axes.ravel(), susc.CRITERIA):
    show(layers[k], f"{titles[k]} (1 = très exposé)", "YlOrRd", 0, 1, ax=ax)
plt.tight_layout()

## 3. Les critères se ressemblent-ils ?

In [12]:
stack = pd.DataFrame({k: layers[k][valid] for k in susc.CRITERIA})
corr = stack.corr()
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(4), corr.columns, rotation=45)
ax.set_yticks(range(4), corr.columns)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{corr.iloc[i, j]:+.2f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()

**Lecture.** Pente et TWI sont corrélés (+0,6) : un terrain plat concentre l'eau. Hauteur et pente le sont aussi (+0,5) : près du fleuve, le terrain est plat. Les critères ne sont donc pas indépendants, ce qui limite l'intérêt d'ajouter des critères du même type.

## 4. Sensibilité aux poids

In [13]:
scores = {name: susc.weighted_score(layers, w) for name, w in susc.SCENARIOS.items()}
ref = scores[susc.MAIN_SCENARIO][valid]
pd.DataFrame({
    "corrélation avec poids égaux": {n: np.corrcoef(ref, s[valid])[0, 1] for n, s in scores.items()}
}).round(3)

,corrélation avec poids égaux
poids_egaux,1.000
fleuve_dominant,0.988
reseau_dominant,0.974
relief_dominant,0.974
convergence_dominante,0.953


In [14]:
robust = load("susceptibility_robust_utm.tif")
main = load("susceptibility_bamako_utm.tif")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
show(main, "Indice combiné (poids égaux)", "YlOrRd", 0, 1, ax=axes[0])
show(robust, "Robustesse : part des 100 tirages de poids où le pixel est dans le top 20 %", "Blues", 0, 1, ax=axes[1])
plt.tight_layout()
print(f"{100 * np.nanmean(robust >= 0.9):.1f} % des pixels sont dans le top 20 % dans au moins 90 % des tirages.")

7.2 % des pixels sont dans le top 20 % dans au moins 90 % des tirages.


**Lecture.** Les cartes issues de scénarios de poids très différents restent corrélées à 0,95–0,99 : le choix des poids change peu la carte d'ensemble. Seule une minorité de pixels est « très exposée » quels que soient les poids : c'est la zone la plus fiable.

## 5. Deux composantes lisibles

In [15]:
fluvial, local = load("susceptibility_fluvial_utm.tif"), load("susceptibility_local_utm.tif")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
show(fluvial, "Composante fluviale (hauteur + distance)", "YlOrRd", 0, 1, ax=axes[0])
show(local, "Composante locale (pente + TWI) — NON validée", "YlOrRd", 0, 1, ax=axes[1])
plt.tight_layout()
print(f"Corrélation fluviale / locale : {np.corrcoef(fluvial[valid], local[valid])[0, 1]:+.2f}")

Corrélation fluviale / locale : +0.28


## 6. Les étiquettes d'eau Sentinel-1 (2024)

Pour chaque date de la saison des pluies, un pixel est « eau nouvelle » s'il est **sombre** (VV < −18 dB), a **baissé d'au moins 3 dB** par rapport à la référence sèche de mars-avril, n'était **pas déjà sombre** en saison sèche, et fait partie d'un groupe d'au moins 8 pixels. Les seuils sont des valeurs courantes de la littérature, **non calibrées sur Bamako**.

Pour tout recalculer : `python -m src.features.flood_labels` (quelques minutes). Ici on lit le résultat.

In [16]:
from src.features import flood_labels as fl

print(f"Seuils : eau < {fl.T_WATER} dB, baisse > {abs(fl.T_CHANGE)} dB, groupe >= {fl.MIN_PIXELS} pixels, pente <= {fl.MAX_SLOPE_DEG}°")
count = load("flood_s1_count_utm.tif")
count[count < 0] = np.nan
urban = load("urban_mask_utm.tif") > 0
area = 30 * 30 / 1e6

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
show(count, "Nombre de dates avec eau nouvelle (sur 5)", "YlGnBu", 0, 5, ax=axes[0])
axes[1].hist(count[count >= 1], bins=np.arange(0.5, 6.5, 1), rwidth=0.8, color="#4a7ebb")
axes[1].set_title("Pixels détectés, par nombre de dates")
plt.tight_layout()
print(f"Au moins 1 date : {np.nansum(count >= 1) * area:.1f} km² | au moins 2 dates : {np.nansum(count >= 2) * area:.1f} km²")
print(f"Part des pixels détectés en zone bâtie : {100 * np.mean(urban[count >= 1]):.0f} % "
      f"(la zone bâtie couvre {100 * np.mean(urban[~np.isnan(count)]):.0f} % de la carte)")

Seuils : eau < -18.0 dB, baisse > 3.0 dB, groupe >= 8 pixels, pente <= 8.0°
Au moins 1 date : 16.2 km² | au moins 2 dates : 14.3 km²
Part des pixels détectés en zone bâtie : 6 % (la zone bâtie couvre 49 % de la carte)


**Lecture.** Les pixels détectés suivent le lit du Niger : on voit la **montée du fleuve et sa plaine d'inondation**, pas l'inondation des quartiers (6 % seulement en zone bâtie, contre 49 % de la carte). Ces étiquettes sont **dérivées d'un capteur, sans vérification sur le terrain**, et le radar voit mal l'eau entre les bâtiments. Un pixel à 0 veut dire « non détecté », pas « jamais inondé ».

La suite est dans le notebook 03 : que valent l'indice et un modèle face à ces étiquettes ?